# 15 · Deploy and query a model serving endpoint

**Prerequisites:** 00, 14 with registration enabled; model-serving permissions and available quota.

**Execution:** optional. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Understand registration versus deployment; create an optional CPU endpoint; query with tensor input.


## Step 1 — Inspect deployment settings

Run notebook 14 with REGISTER=True first. Copy the registered three-part model name and its numeric version here. Endpoint creation consumes workspace resources; the default dry run prints the plan.


In [ ]:
CREATE_ENDPOINT=False
MODEL_NAME=''
MODEL_VERSION=''
ENDPOINT_NAME='ai-engineering-churn-demo'
print({'create':CREATE_ENDPOINT,'model':MODEL_NAME,'version':MODEL_VERSION,'endpoint':ENDPOINT_NAME})


**Check your result:** No endpoint is created in the default configuration.


## Step 2 — Create and wait

Use an existing compatible registered sklearn model from lab 14. If the endpoint name exists, use a new name or manage that existing course endpoint in Serving UI; do not overwrite it blindly.


In [ ]:
if CREATE_ENDPOINT:
    assert len(MODEL_NAME.split('.'))==3 and MODEL_VERSION.isdigit()
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
    w=WorkspaceClient()
    endpoint=w.serving_endpoints.create_and_wait(name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=[ServedEntityInput(
            entity_name=MODEL_NAME,entity_version=MODEL_VERSION,
            workload_size='Small',scale_to_zero_enabled=True)]))
    print('Endpoint ready:',endpoint.name)
else:
    print('Dry run: registration alone does not start a serving endpoint.')


**Check your result:** Enabled mode waits for endpoint readiness; quota, permissions and package support still apply.


## Step 3 — Query and inspect latency

This sends one numeric tensor with ten features, matching the lab 14 signature. Use production-valid feature ranges in a real application.


In [ ]:
if CREATE_ENDPOINT:
    import time
    started=time.perf_counter()
    response=w.serving_endpoints.query(name=ENDPOINT_NAME,inputs=[[0.0]*10])
    print('Response:',response.as_dict())
    print('Latency seconds:',time.perf_counter()-started)
else:
    print('Query skipped; enable creation only after setting model/version.')


**Check your result:** The enabled response includes one prediction. First requests can be slower after scale-to-zero.


## Practice

Measure warm versus cold latency. After the lab, delete only your course endpoint from Serving UI to release quota.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Collect repeated timing measurements and report median/p95, failure rate and request shape. Deleting an endpoint is separate from deleting the registered model.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
